# LLMOps Assignment 2 — API Cost, Latency, and Prompt Version Management
**Course:** MLOps | **Scenario:** EduAssist AI  
This notebook calls the local Ollama model `llama3.1:8b` for the same five academic questions under three prompt versions. It records request latency and token-based cost, registers prompts in MLflow, logs aggregate metrics, checks JSON validity for v3, and assigns a `production` alias based on the measured results.

> **Runtime setup:** Install and start Ollama, then run `ollama pull llama3.1:8b`. No cloud API key is required. Local token local token cost is recorded as zero by default; VM/GPU/CPU and electricity costs are excluded. The notebook intentionally does not include fabricated results; generated CSVs reflect actual runs.

## Runtime setup (run in terminal before starting the notebook)\n```bash\nollama pull llama3.1:8b\nollama serve  # only if the Ollama service is not already running\npip install -U ollama mlflow pandas\n```\nKeep Ollama running while executing the notebook. Set `OLLAMA_HOST` if Ollama is on another machine.\n

In [1]:
import os, json, time, statistics, math
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import mlflow
import ollama



OUTPUT_DIR = Path(".")
MODEL = os.getenv("OLLAMA_MODEL", "llama3.1:8b")
OLLAMA_HOST = os.getenv("OLLAMA_HOST", "http://localhost:11434")
# Local Ollama inference has no per-token API charge. These optional rates default to zero.
# They can be changed to internal chargeback rates, if required; compute/electricity costs are excluded.
INPUT_PRICE_PER_MILLION = float(os.getenv("LOCAL_INPUT_PRICE_PER_MILLION", "0"))
OUTPUT_PRICE_PER_MILLION = float(os.getenv("LOCAL_OUTPUT_PRICE_PER_MILLION", "0"))
MONTHLY_REQUESTS = 10_000
EXPERIMENT_NAME = "eduassist_llmops_assignment"

KEEP_ALIVE = "5m"
NUM_CTX = 1024
MAX_OUTPUT_TOKENS = 180

OLLAMA_OPTIONS = {
    "temperature": 0,
    "num_ctx": NUM_CTX,
    "num_predict": MAX_OUTPUT_TOKENS,
    "num_batch": 128,
    "num_thread": 4
}


client = ollama.Client(host=OLLAMA_HOST)
print("Model:", MODEL)
print("Ollama host:", OLLAMA_HOST)
print("Local token pricing (USD per 1M tokens): input =", INPUT_PRICE_PER_MILLION,
      ", output =", OUTPUT_PRICE_PER_MILLION)
print("MLflow version:", mlflow.__version__)

# Confirm Ollama is reachable and the requested model is already downloaded.
available_models = [m.model for m in client.list().models]
if not any(name.split(":")[0] == MODEL.split(":")[0] and
           (":" not in MODEL or name.endswith(MODEL.split(":")[-1]))
           for name in available_models):
    raise RuntimeError(
        f"{MODEL} is not available in Ollama. Run `ollama pull {MODEL}` in a terminal, "
        "then rerun this cell."
    )
print("Ollama model is available.")


Model: llama3.1:8b
Ollama host: http://localhost:11434
Local token pricing (USD per 1M tokens): input = 0.0 , output = 0.0
MLflow version: 3.18.0
Ollama model is available.


In [2]:
questions = [
    {"question_id": "Q1", "question": "What is the difference between supervised and unsupervised learning?"},
    {"question_id": "Q2", "question": "Explain overfitting in machine learning and give two ways to reduce it."},
    {"question_id": "Q3", "question": "What is the purpose of a train-test split when building a machine learning model?"},
    {"question_id": "Q4", "question": "Explain precision and recall, and describe when each metric is useful."},
    {"question_id": "Q5", "question": "What is data drift, and why should a deployed machine learning system monitor it?"}
]

PROMPTS = {
    "v1": "Answer the following student question clearly.\nQuestion: {{question}}",
    "v2": "You are an academic assistant.\nAnswer the following student question in a clear, concise, and accurate manner.\nUse simple language suitable for undergraduate students.\nQuestion: {{question}}",
    "v3": '''You are an academic assistant.
Answer the student question and return ONLY valid JSON.
Do not include markdown, explanations outside JSON, or code fences.
Use this exact JSON schema:
{
 "answer": "string",
 "confidence": "high | medium | low",
 "key_points": ["string"]
}
Question: {{question}}'''
}
COMMIT_MESSAGES = {
    "v1": "Initial baseline academic QA prompt",
    "v2": "Add academic-assistant role and undergraduate-friendly style guidance",
    "v3": "Require schema-constrained JSON output for downstream validation"
}
questions

[{'question_id': 'Q1',
  'question': 'What is the difference between supervised and unsupervised learning?'},
 {'question_id': 'Q2',
  'question': 'Explain overfitting in machine learning and give two ways to reduce it.'},
 {'question_id': 'Q3',
  'question': 'What is the purpose of a train-test split when building a machine learning model?'},
 {'question_id': 'Q4',
  'question': 'Explain precision and recall, and describe when each metric is useful.'},
 {'question_id': 'Q5',
  'question': 'What is data drift, and why should a deployed machine learning system monitor it?'}]

## Task 1 and Task 2 — local Ollama requests, latency, tokens, and costs
We run each prompt version against the same five questions (15 requests total). Token usage is taken from the API response. Failed calls are recorded with a status and error, but their missing token counts are not treated as zero-cost successes.\nToken counts are read from Ollama's `prompt_eval_count` and `eval_count`. The default local per-token API cost is USD 0; hardware, VM/GPU/CPU, and electricity costs are not included.\n

In [ ]:
def render_prompt(template: str, question: str) -> str:
    return template.replace("{{question}}", question)


def cost_usd(input_tokens, output_tokens):
    return (
        (input_tokens / 1_000_000 * INPUT_PRICE_PER_MILLION)
        + (output_tokens / 1_000_000 * OUTPUT_PRICE_PER_MILLION)
    )


# --------------------------------------------------
# Run local Ollama evaluations sequentially
# --------------------------------------------------

records = []
raw_outputs = {}

for version, template in PROMPTS.items():
    for item in questions:
        started = datetime.now(timezone.utc)
        t0 = time.perf_counter()

        status = "success"
        output = ""
        input_tokens = None
        output_tokens = None
        error = ""

        prompt = render_prompt(template, item["question"])

        try:
            response = client.generate(
                model=MODEL,
                prompt=prompt,
                options={
                    "temperature": 0,
                    "num_ctx": NUM_CTX,
                    "num_predict": MAX_OUTPUT_TOKENS,
                    "num_batch": 128,
                    "num_thread": 4
                },
                keep_alive=KEEP_ALIVE
            )

            output = response.get("response", "") or ""
            # Ollama reports prompt and generated token counts in these fields.
            input_tokens = response.get("prompt_eval_count")
            output_tokens = response.get("eval_count")
            input_tokens = int(input_tokens) if input_tokens is not None else None
            output_tokens = int(output_tokens) if output_tokens is not None else None

            raw_outputs[(version, item["question_id"])] = output

        except Exception as exc:
            status = "error"
            error = f"{type(exc).__name__}: {exc}"
            print(f"Ollama Error | Version: {version} | "
                  f"Question: {item['question_id']} | {error}")

        elapsed = time.perf_counter() - t0
        ended = datetime.now(timezone.utc)

        total_tokens = (
            input_tokens + output_tokens
            if input_tokens is not None and output_tokens is not None
            else None
        )
        input_cost = (
            input_tokens / 1_000_000 * INPUT_PRICE_PER_MILLION
            if input_tokens is not None else None
        )
        output_cost = (
            output_tokens / 1_000_000 * OUTPUT_PRICE_PER_MILLION
            if output_tokens is not None else None
        )
        request_cost = (
            cost_usd(input_tokens, output_tokens)
            if input_tokens is not None and output_tokens is not None
            else None
        )

        records.append({
            "question_id": item["question_id"],
            "prompt_version": version,
            "model_name": MODEL,
            "start_time_utc": started.isoformat(),
            "end_time_utc": ended.isoformat(),
            "latency_seconds": round(elapsed, 6),
            "response_status": status,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "total_tokens": total_tokens,
            "input_cost_usd": input_cost,
            "output_cost_usd": output_cost,
            "cost_per_request_usd": request_cost,
            "error": error,
            "response_text": output
        })

latency_df = pd.DataFrame(records)
latency_df.to_csv(OUTPUT_DIR / "latency_tokens_cost.csv", index=False)

display(latency_df[[
    "question_id", "prompt_version", "model_name", "latency_seconds",
    "response_status", "input_tokens", "output_tokens", "total_tokens",
    "cost_per_request_usd"
]])

print("\\nEvaluation Summary")
print("------------------")
print("Total requests:", len(latency_df))
print("Successful requests:", (latency_df["response_status"] == "success").sum())
print("Failed requests:", (latency_df["response_status"] == "error").sum())
print("Average latency:", round(latency_df["latency_seconds"].mean(), 4), "seconds")
print("Total local token cost (USD):",
      round(latency_df["cost_per_request_usd"].sum(), 8))
print("Saved:", OUTPUT_DIR / "latency_tokens_cost.csv")


In [ ]:
successful = latency_df.loc[latency_df["response_status"] == "success"].copy()
latencies = successful["latency_seconds"].dropna()
latency_summary = pd.DataFrame([{
    "count": int(latencies.count()),
    "mean_seconds": float(latencies.mean()) if len(latencies) else None,
    "median_seconds": float(latencies.median()) if len(latencies) else None,
    "min_seconds": float(latencies.min()) if len(latencies) else None,
    "max_seconds": float(latencies.max()) if len(latencies) else None,
    "std_seconds_sample": float(latencies.std(ddof=1)) if len(latencies) > 1 else 0.0,
}])
display(latency_summary)
if len(successful):
    slowest = successful.loc[successful["latency_seconds"].idxmax()]
    print(f"Slowest successful request: {slowest['question_id']} / {slowest['prompt_version']} at {slowest['latency_seconds']:.3f}s.")
    print("Possible reasons include network/service variability, request queueing, prompt length, and generated output length. One observation alone cannot identify the cause.")

## Task 3 — MLflow prompt registry and evaluation
The notebook registers v1–v3 under one prompt name, logs an MLflow run for each version, and computes the required aggregate metrics. `valid_json` is the fraction of successful responses that parse as JSON with the required keys and value types. For v1/v2, JSON validity is still measured but is not a required output property; the comparison table will show their observed rates.

In [ ]:
MLFLOW_DB = Path("mlflow.db").resolve()

mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB}")
mlflow.set_experiment(EXPERIMENT_NAME)

print("MLflow Tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", EXPERIMENT_NAME)


PROMPT_NAME = "eduassist_academic_qa"

# Prompt Registry is available in modern MLflow releases. If the server/client
# is older, upgrade MLflow before running this section.
registered = {}
for version, template in PROMPTS.items():
    registered[version] = mlflow.genai.register_prompt(
        name=PROMPT_NAME,
        template=template,
        commit_message=COMMIT_MESSAGES[version],
        tags={"assignment": "llmops_assignment_2", "prompt_version_label": version}
    )
    print(version, "registered as MLflow prompt version", registered[version].version)

In [ ]:
def is_valid_eduassist_json(text):
    try:
        obj = json.loads(text)
        if not isinstance(obj, dict):
            return False
        if not isinstance(obj.get("answer"), str):
            return False
        if obj.get("confidence") not in {"high", "medium", "low"}:
            return False
        if not isinstance(obj.get("key_points"), list) or not all(isinstance(x, str) for x in obj["key_points"]):
            return False
        return True
    except (json.JSONDecodeError, TypeError):
        return False

evaluation_rows = []
for version in PROMPTS:
    subset = latency_df[(latency_df["prompt_version"] == version) & (latency_df["response_status"] == "success")].copy()
    validity = [is_valid_eduassist_json(raw_outputs.get((version, qid), "")) for qid in subset["question_id"]]
    avg_latency = subset["latency_seconds"].mean() if len(subset) else float("nan")
    avg_input = subset["input_tokens"].dropna().mean() if len(subset) else float("nan")
    avg_output = subset["output_tokens"].dropna().mean() if len(subset) else float("nan")
    avg_cost = subset["cost_per_request_usd"].dropna().mean() if len(subset) else float("nan")
    valid_json_rate = (sum(validity) / len(validity)) if validity else 0.0
    evaluation_rows.append({
        "prompt_version": version,
        "mlflow_prompt_version": str(registered[version].version),
        "avg_latency_seconds": avg_latency,
        "avg_input_tokens": avg_input,
        "avg_output_tokens": avg_output,
        "avg_cost_per_request_usd": avg_cost,
        "valid_json": valid_json_rate,
        "successful_responses": int(len(subset)),
        "recommended": False
    })

eval_df = pd.DataFrame(evaluation_rows)
display(eval_df)


In [ ]:
# Log metrics to MLflow. Missing values are omitted rather than logged as NaN.
for _, row in eval_df.iterrows():
    version = row["prompt_version"]
    with mlflow.start_run(run_name=f"prompt_{version}") as run:
        mlflow.log_param("prompt_version_label", version)
        mlflow.log_param("mlflow_prompt_version", row["mlflow_prompt_version"])
        mlflow.log_param("model_name", MODEL)
        mlflow.log_param("input_price_per_million_usd", INPUT_PRICE_PER_MILLION)
        mlflow.log_param("output_price_per_million_usd", OUTPUT_PRICE_PER_MILLION)
        for metric in ["avg_latency_seconds", "avg_input_tokens", "avg_output_tokens", "avg_cost_per_request_usd", "valid_json"]:
            value = row[metric]
            if pd.notna(value):
                mlflow.log_metric(metric, float(value))
        mlflow.log_metric("successful_responses", float(row["successful_responses"]))

# Choose a production candidate transparently: prefer JSON-valid v3 if it achieves
# 100% validity; otherwise choose the highest valid_json rate, then lower cost,
# then lower latency. This rule is deterministic for the observed results.
ranked = eval_df.copy()
ranked["valid_json_sort"] = ranked["valid_json"].fillna(-1)
ranked["cost_sort"] = ranked["avg_cost_per_request_usd"].fillna(float("inf"))
ranked["latency_sort"] = ranked["avg_latency_seconds"].fillna(float("inf"))
ranked = ranked.sort_values(["valid_json_sort", "cost_sort", "latency_sort"], ascending=[False, True, True])
winner = ranked.iloc[0]["prompt_version"]
eval_df.loc[eval_df["prompt_version"] == winner, "recommended"] = True

# Assign the production alias to the selected registered prompt version.
winner_mlflow_version = int(eval_df.loc[eval_df["prompt_version"] == winner, "mlflow_prompt_version"].iloc[0])
mlflow.genai.set_prompt_alias(PROMPT_NAME, alias="production", version=winner_mlflow_version)

eval_df.to_csv("prompt_evaluation_results.csv", index=False)
print(f"Recommended version: {winner}; assigned MLflow alias '{PROMPT_NAME}@production' -> version {winner_mlflow_version}.")
display(eval_df)
print("Saved prompt_evaluation_results.csv")

In [ ]:
# Summary tables for the report
cost_summary = successful.groupby("prompt_version", as_index=False).agg(
    successful_requests=("question_id", "count"),
    avg_cost_per_request_usd=("cost_per_request_usd", "mean"),
    avg_input_tokens=("input_tokens", "mean"),
    avg_output_tokens=("output_tokens", "mean"),
    avg_latency_seconds=("latency_seconds", "mean"),
)
cost_summary["projected_monthly_cost_10000_usd"] = cost_summary["avg_cost_per_request_usd"] * MONTHLY_REQUESTS
display(cost_summary)

print("Monthly projection formula: average cost per successful request × 10,000.")
print("Pricing is an estimate in USD, based on the configured model rates and API usage counts.")

## Required interpretation and screenshots
- **Slowest request:** use the row with maximum latency from `latency_tokens_cost.csv`; discuss plausible causes only, not a definitive diagnosis.
- **Cost optimization:** keep answers concise with a response token cap; use caching for stable instructions/context; consider a smaller model for simple questions after quality evaluation.
- **Prompt diff:** in MLflow UI, open `eduassist_academic_qa` and compare versions 1, 2, and 3. The prompt text and commit messages are the version evidence.
- **Screenshots to capture manually:** experiment runs and metrics; Prompt Registry showing versions; prompt comparison/diff; `production` alias on the selected version.
- **Report:** populate the report template with actual output from the CSVs. Never claim the code was executed successfully unless you ran it and verified the outputs.

In [ ]:
import mlflow

mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000"))

experiment_name = "eduassist_llmops_assignment"
experiment = mlflow.get_experiment_by_name(experiment_name)

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", experiment.name if experiment else "Not found")
print("Experiment ID:", experiment.experiment_id if experiment else "Not found")


In [ ]:
from mlflow import MlflowClient

mlflow_client = MlflowClient()
prompts = mlflow_client.search_prompts(
    filter_string="name = 'eduassist_academic_qa'"
)

print("Registered prompts:")
for prompt in prompts:
    print(prompt.name, prompt.version)


In [ ]:
import pandas as pd

df = pd.read_csv("prompt_evaluation_results.csv")

print("Rows:", len(df))
print("\nColumns:")
print(df.columns.tolist())

display(df.head(10))

In [ ]:
import pandas as pd

df = pd.read_csv("latency_tokens_cost.csv")

print("Total API calls:", len(df))
display(df.head(10))

if "status" in df.columns:
    print("\nAPI status:")
    print(df["status"].value_counts(dropna=False))

if "error" in df.columns:
    print("\nAPI errors:")
    print(df["error"].dropna().tolist()[:10])